In [0]:
import pandas as pd

In [0]:
spark.sql("use catalog proyecto_final_prueba")

In [0]:
catalog = spark.sql("select current_catalog()").first()[0]
schema = "gold"
table = "dim_tiempo"

In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")

In [0]:
spark.sql(f"drop table if exists {catalog}.{schema}.{table}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
  id_tiempo BIGINT,
  fecha_hora_local TIMESTAMP,
  fecha DATE,
  anio INT,
  mes INT,
  dia INT,
  hora INT,
  dia_semana STRING,
  es_fin_de_semana BOOLEAN
)
""")

In [0]:
df_silver = spark.table(f"{catalog}.silver.weather").toPandas()
df_silver


In [0]:
dim = pd.DataFrame({'fecha_hora_local': df_silver['fecha_hora_local'].unique()})
dim

In [0]:
dim = dim.sort_values('fecha_hora_local').reset_index(drop=True)
dim['id_tiempo'] =dim['fecha_hora_local'].dt.strftime('%Y%m%d%H').astype(int)
dim

In [0]:
dim['fecha'] = dim['fecha_hora_local'].dt.date
dim['anio'] = dim['fecha_hora_local'].dt.year
dim['mes'] = dim['fecha_hora_local'].dt.month
dim['dia'] = dim['fecha_hora_local'].dt.day
dim['hora'] = dim['fecha_hora_local'].dt.hour
dim

In [0]:
dias_espanol = {
    0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 
    3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'
}
dim['dia_semana'] = dim['fecha_hora_local'].dt.dayofweek.map(dias_espanol)
dim

In [0]:
dim['es_fin_de_semana'] = dim['fecha_hora_local'].dt.dayofweek >= 5
columnas_dim = [
    'id_tiempo', 'fecha_hora_local', 'fecha', 'anio', 
    'mes', 'dia', 'hora', 'dia_semana', 'es_fin_de_semana'
]
dim = dim[columnas_dim]
dim

In [0]:
df_spark = spark.createDataFrame(dim)
df_spark.display()

In [0]:
df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{table}")